# Exercise - Treasury Arbitrage

#### Notation Commands

$$\newcommand{\Black}{\mathcal{B}}
\newcommand{\Blackcall}{\Black_{\mathrm{call}}}
\newcommand{\Blackput}{\Black_{\mathrm{put}}}
\newcommand{\EcondS}{\hat{S}_{\mathrm{conditional}}}
\newcommand{\Efwd}{\mathbb{E}^{T}}
\newcommand{\Ern}{\mathbb{E}^{\mathbb{Q}}}
\newcommand{\Tfwd}{T_{\mathrm{fwd}}}
\newcommand{\Tunder}{T_{\mathrm{bond}}}
\newcommand{\accint}{A}
\newcommand{\carry}{\widetilde{\cpn}}
\newcommand{\cashflow}{C}
\newcommand{\convert}{\phi}
\newcommand{\cpn}{c}
\newcommand{\ctd}{\mathrm{CTD}}
\newcommand{\disc}{Z}
\newcommand{\done}{d_{1}}
\newcommand{\dt}{\Delta t}
\newcommand{\dtwo}{d_{2}}
\newcommand{\flatvol}{\sigma_{\mathrm{flat}}}
\newcommand{\flatvolT}{\sigma_{\mathrm{flat},T}}
\newcommand{\float}{\mathrm{flt}}
\newcommand{\freq}{m}
\newcommand{\futprice}{\mathcal{F}(t,T)}
\newcommand{\futpriceDT}{\mathcal{F}(t+h,T)}
\newcommand{\futpriceT}{\mathcal{F}(T,T)}
\newcommand{\futrate}{\mathscr{f}}
\newcommand{\fwdprice}{F(t,T)}
\newcommand{\fwdpriceDT}{F(t+h,T)}
\newcommand{\fwdpriceT}{F(T,T)}
\newcommand{\fwdrate}{f}
\newcommand{\fwdvol}{\sigma_{\mathrm{fwd}}}
\newcommand{\fwdvolTi}{\sigma_{\mathrm{fwd},T_i}}
\newcommand{\grossbasis}{B}
\newcommand{\hedge}{\Delta}
\newcommand{\ivol}{\sigma_{\mathrm{imp}}}
\newcommand{\logprice}{p}
\newcommand{\logyield}{y}
\newcommand{\mat}{(n)}
\newcommand{\nargcond}{d_{1}}
\newcommand{\nargexer}{d_{2}}
\newcommand{\netbasis}{\tilde{\grossbasis}}
\newcommand{\normcdf}{\mathcal{N}}
\newcommand{\notional}{K}
\newcommand{\pfwd}{P_{\mathrm{fwd}}}
\newcommand{\pnl}{\Pi}
\newcommand{\price}{P}
\newcommand{\probexer}{\hat{\mathcal{P}}_{\mathrm{exercise}}}
\newcommand{\pvstrike}{K^*}
\newcommand{\refrate}{r^{\mathrm{ref}}}
\newcommand{\rrepo}{r^{\mathrm{repo}}}
\newcommand{\spotrate}{r}
\newcommand{\spread}{s}
\newcommand{\strike}{K}
\newcommand{\swap}{\mathrm{sw}}
\newcommand{\swaprate}{\cpn_{\swap}}
\newcommand{\tbond}{\mathrm{fix}}
\newcommand{\ttm}{\tau}
\newcommand{\value}{V}
\newcommand{\vega}{\nu}
\newcommand{\years}{\tau}
\newcommand{\yearsACT}{\tau_{\mathrm{act/360}}}
\newcommand{\yield}{Y}$$

# 1. Treasury Arbitrage

Consider the following market data as of `Dec 29, 2023`.

The table below shows two Treasury securities, a T-note and a T-bond. They mature on the same date.

In [10]:
import pandas as pd
from pandas import IndexSlice

summary = pd.DataFrame(index=[],columns = [207391,204095],dtype=float)
summary.loc['issue date'] = ['2019-08-15','1999-08-15']
summary.loc['maturity date'] = ['2029-08-15','2029-08-15']
summary.loc['coupon rate'] = [.01625, .06125]
summary.loc['clean price'] = [89.03125,111.0391]
summary.loc['accrued interest'] = [.6005, 2.2636]
summary.loc['ytm'] = [.037677, .038784]

summary.style.format("{:.2%}", subset=IndexSlice[["ytm","coupon rate"], :]).format("{:.2f}", subset=IndexSlice[["accrued interest","clean price"], :])

,207391,204095
issue date,2019-08-15,1999-08-15
maturity date,2029-08-15,2029-08-15
coupon rate,1.62%,6.12%
clean price,89.03,111.04
accrued interest,0.60,2.26
ytm,3.77%,3.88%


As usual, the quotes are per $100 face value.

### 1.1.

Which bond would you go long, and which bond would you short?

Explain your reasoning.

Go Long the T-bond (ID=204095) and Short the T-note (ID=207391).

They mature on the same date, so a first-pass relative-value comparison is yield:
- T-bond YTM = 3.8784%
- T-note YTM = 3.7677%

The higher-YTM bond is cheaper (offers more yield for the same maturity date), so it is the long leg.
The lower-YTM bond is richer, so it is the short leg.

### 1.2.

Explain how you would finance the trade? Be specific in explaining how you would raise the cash for the long position and how you would achieve the short position.

Finance the Long (T-bond) via repo: borrow cash secured by the bond (you post the bond as collateral),
so your net funding cost is close to the repo rate.

Implement the Short (T-note) by borrowing the security (via securities lending / repo special),
selling it in the market, and posting collateral (cash or Treasuries) with the lender.
The short sale proceeds are typically held as part of the collateral arrangement and you pay the borrow/repo rate.

Net: you are long one Treasury financed in repo and short another Treasury via borrow-and-sell,
aiming to profit if the yield/spread between them converges.

### 1.3. 

What are the risks of this trade? Is it an arbitrage in the short-term or in the long-term? Explain.

Risks:
1) Spread / basis risk: the yield spread between the two issues can widen rather than converge.
2) Curve / convexity mismatch: even with same maturity date, coupons differ, so duration/convexity differ.
   Without DV01-hedging, you have residual rate risk.
3) Financing risk: repo rates can change; one issue may go “special” (hard-to-borrow) which can be costly.
4) Liquidity/market impact: bid-ask and liquidity differences can affect entry/exit.
5) Model/assumption risk: treating the “fair” spread as stable can fail.

This is not a risk-free arbitrage. It is a *relative-value convergence trade*.
The convergence (if it happens) is typically a shorter-to-medium horizon micropricing/financing effect,
not a guaranteed long-term arbitrage.

### 1.4.

Suppose that on `2024-02-15`, immediately after the coupon is paid out, we observe the prices are `87` and `113`, respectively.
* Approximate time-to-maturity as 5.5 years.
* Note that the coupon was just paid, so there is no accrued interest.

Calculate the new YTMs for the two bonds.

In [11]:
def bond_price(ttm_years, coupon_rate, ytm, frequency=2, face_value=100.0):
    """
    Plain-vanilla fixed coupon bond price per $face_value, assuming:
    - coupons paid 'frequency' times per year
    - ytm is nominal annual yield compounded at 'frequency'
    - settlement is just after coupon if accrued interest is 0 (dirty=clean)
    """
    n = int(round(ttm_years * frequency))
    c = coupon_rate * face_value / frequency
    df = [(1.0 + ytm / frequency) ** (-k) for k in range(1, n + 1)]
    return c * sum(df) + face_value * df[-1]

def bond_ytm_bisect(ttm_years, coupon_rate, market_price, frequency=2, face_value=100.0,
                    lo=0.0, hi=0.30, tol=1e-12, max_iter=200):
    """
    Robust YTM solver via bisection. Assumes price decreases with yield over bracket.
    """
    def f(y):
        return bond_price(ttm_years, coupon_rate, y, frequency, face_value) - market_price

    flo, fhi = f(lo), f(hi)
    if flo == 0:
        return lo
    if fhi == 0:
        return hi
    if flo * fhi > 0:
        raise ValueError(
            f"YTM not bracketed: f(lo)={flo:.6f}, f(hi)={fhi:.6f}. "
            "Widen hi or adjust lo."
        )

    for _ in range(max_iter):
        mid = 0.5 * (lo + hi)
        fmid = f(mid)
        if abs(fmid) < tol or (hi - lo) < tol:
            return mid
        if flo * fmid <= 0:
            hi, fhi = mid, fmid
        else:
            lo, flo = mid, fmid

    return 0.5 * (lo + hi)

ttm_2024_02_15 = 5.5

ytm_note_2024_02_15 = bond_ytm_bisect(ttm_2024_02_15, 0.01625, 87.0)
ytm_bond_2024_02_15 = bond_ytm_bisect(ttm_2024_02_15, 0.06125, 113.0)

print(f"New YTM for T-note (207391) @ 2024-02-15: {ytm_note_2024_02_15:.6%}")
print(f"New YTM for T-bond (204095) @ 2024-02-15: {ytm_bond_2024_02_15:.6%}")

New YTM for T-note (207391) @ 2024-02-15: 4.304703%
New YTM for T-bond (204095) @ 2024-02-15: 3.505591%


### 1.5.

Suppose that on `2024-08-15` we observe the YTMs are now 4.65% and 4.70%, respectively. 

Note that this observation is immediately **after** the bonds pay out coupons.

* Calculate the prices of both bonds.
* Compare this to the **dirty** prices inferred from the table above.


In [12]:
ttm_2024_08_15 = 5.0

price_note_2024_08_15 = bond_price(ttm_2024_08_15, 0.01625, 0.0465)
price_bond_2024_08_15 = bond_price(ttm_2024_08_15, 0.06125, 0.0470)

print(f"Price of T-note (207391) at YTM 4.65% on 2024-08-15: {price_note_2024_08_15:.4f}")
print(f"Price of T-bond (204095) at YTM 4.70% on 2024-08-15: {price_bond_2024_08_15:.4f}")

# FIX: You cannot compare Aug-2024 prices to Dec-2023 dirty prices directly (different dates).
# We'll compute and display both Dec-29 dirty prices and Aug-15 clean/dirty (same just after coupon).

dirty_note_2023_12_29 = summary.loc["clean price", 207391] + summary.loc["accrued interest", 207391]
dirty_bond_2023_12_29 = summary.loc["clean price", 204095] + summary.loc["accrued interest", 204095]

print("\nDirty prices as of Dec 29, 2023 (from table):")
print(f"  T-note dirty (Dec 29, 2023): {dirty_note_2023_12_29:.4f}")
print(f"  T-bond dirty (Dec 29, 2023): {dirty_bond_2023_12_29:.4f}")

print("\nPrices as of Aug 15, 2024 immediately after coupon (dirty = clean):")
print(f"  T-note price (Aug 15, 2024): {price_note_2024_08_15:.4f}")
print(f"  T-bond price (Aug 15, 2024): {price_bond_2024_08_15:.4f}")

Price of T-note (207391) at YTM 4.65% on 2024-08-15: 86.6420
Price of T-bond (204095) at YTM 4.70% on 2024-08-15: 106.2845

Dirty prices as of Dec 29, 2023 (from table):
  T-note dirty (Dec 29, 2023): 89.6317
  T-bond dirty (Dec 29, 2023): 113.3027

Prices as of Aug 15, 2024 immediately after coupon (dirty = clean):
  T-note price (Aug 15, 2024): 86.6420
  T-bond price (Aug 15, 2024): 106.2845


### 1.6.

Suppose that you have a position...
* long 1 unit of T-bond (ID=`204095`).
* short 1 unit of T-note (ID=`207391`).

(Note that we're not balancing the dollars or risk. Just assume long-short one unit of each.)


Calculate the value of this long-short position using the 
* prices in the given table
* prices from `1.5`

How did the value of the position change? 

In [13]:
initial_value_2023_12_29 = dirty_bond_2023_12_29 - dirty_note_2023_12_29
value_2024_08_15 = price_bond_2024_08_15 - price_note_2024_08_15

print("\nLong-short valuation (LONG bond 204095, SHORT note 207391):")
print(f"  Initial value using Dec 29, 2023 dirty prices: {initial_value_2023_12_29:.4f}")
print(f"  Value using Aug 15, 2024 prices from 1.5:      {value_2024_08_15:.4f}")
print(f"  Change in value:                               {value_2024_08_15 - initial_value_2023_12_29:.4f}")


Long-short valuation (LONG bond 204095, SHORT note 207391):
  Initial value using Dec 29, 2023 dirty prices: 23.6710
  Value using Aug 15, 2024 prices from 1.5:      19.6425
  Change in value:                               -4.0285


***